# Матричная факторизация: ALS для рекомендательных систем

Реализуется `ALS` для разреженной матрицы рейтингов и проверяется,
как модель ведёт себя на данных MovieLens.

## Математическая постановка

## Загрузка и подготовка данных

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import time
import warnings


from pathlib import Path

def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures" / "matrix_factorization"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'ml-32m').mkdir(parents=True, exist_ok=True)

 

warnings.filterwarnings('ignore')

DATA_PATH = DATA_DIR / 'ml-32m'

print('Загрузка данных...')
t0 = time.time()

ratings = pd.read_csv(
    DATA_PATH / 'ratings.csv',
    nrows=500_000,
    dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int64'}
)

movies = pd.read_csv(DATA_PATH / 'movies.csv')

print(f'Загрузка завершена за {time.time()-t0:.2f}с')
print(f'Рейтингов: {len(ratings):,}')
print(f'Фильмов в movies.csv: {len(movies):,}')
print()
print('Первые строки ratings:')
ratings.head()

In [ ]:
# Базовая статистика
print('=== Статистика датасета ===')
print(f'Уникальных пользователей: {ratings["userId"].nunique():,}')
print(f'Уникальных фильмов:       {ratings["movieId"].nunique():,}')
print(f'Диапазон рейтингов:       {ratings["rating"].min()} — {ratings["rating"].max()}')
print(f'Средний рейтинг:          {ratings["rating"].mean():.3f}')
print()

# Распределение рейтингов
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Распределение рейтингов')
axes[0].set_xlabel('Рейтинг')
axes[0].set_ylabel('Количество')

ratings.groupby('userId').size().hist(bins=50, ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Количество оценок на пользователя')
axes[1].set_xlabel('Количество оценок')
axes[1].set_ylabel('Количество пользователей')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'als_data_stats.png', dpi=100, bbox_inches='tight')
plt.show()
print('График сохранён: als_data_stats.png')

In [ ]:
# Оставляем только фильмы с минимальным числом оценок (фильтрация холодного старта)
MIN_RATINGS_PER_USER = 5
MIN_RATINGS_PER_MOVIE = 10

print('Фильтрация малоактивных пользователей и редких фильмов...')
print(f'До фильтрации: {len(ratings):,} записей')

# Итеративная фильтрация для сохранения связности
for _ in range(3):
    user_counts = ratings['userId'].value_counts()
    movie_counts = ratings['movieId'].value_counts()
    
    active_users = user_counts[user_counts >= MIN_RATINGS_PER_USER].index
    active_movies = movie_counts[movie_counts >= MIN_RATINGS_PER_MOVIE].index
    
    ratings = ratings[
        ratings['userId'].isin(active_users) &
        ratings['movieId'].isin(active_movies)
    ]

print(f'После фильтрации: {len(ratings):,} записей')
print(f'Пользователей: {ratings["userId"].nunique():,}')
print(f'Фильмов:       {ratings["movieId"].nunique():,}')

In [ ]:
# Создание маппингов userId→idx и movieId→idx
# Нумерация с нуля для матричной индексации

unique_users = sorted(ratings['userId'].unique())
unique_movies = sorted(ratings['movieId'].unique())

user2idx = {uid: idx for idx, uid in enumerate(unique_users)}
idx2user = {idx: uid for uid, idx in user2idx.items()}

movie2idx = {mid: idx for idx, mid in enumerate(unique_movies)}
idx2movie = {idx: mid for mid, idx in movie2idx.items()}

n_users = len(unique_users)
n_movies = len(unique_movies)

print(f'Размер матрицы: {n_users} пользователей × {n_movies} фильмов')
print(f'Плотность матрицы: {len(ratings) / (n_users * n_movies) * 100:.4f}%')

# Добавляем индексы в датафрейм
ratings['user_idx'] = ratings['userId'].map(user2idx)
ratings['movie_idx'] = ratings['movieId'].map(movie2idx)

# Маппинг movieId → название фильма
movie_id_to_title = dict(zip(movies['movieId'], movies['title']))

In [ ]:
# Создание разреженной матрицы scipy.sparse.csr_matrix
# Работаем ТОЛЬКО с наблюдаемыми элементами — никаких нулей для пропусков

row_indices = ratings['user_idx'].values
col_indices = ratings['movie_idx'].values
data_values = ratings['rating'].values.astype(np.float32)

# csr_matrix — эффективная для строчного доступа (обновление U)
R_sparse = sp.csr_matrix(
    (data_values, (row_indices, col_indices)),
    shape=(n_users, n_movies),
    dtype=np.float32
)

# csc_matrix — эффективная для столбцового доступа (обновление V)
R_csc = R_sparse.tocsc()

print(f'Разреженная матрица создана:')
print(f'  Форма:           {R_sparse.shape}')
print(f'  Ненулевых элем.: {R_sparse.nnz:,}')
print(f'  Потребление памяти (CSR): {R_sparse.data.nbytes / 1024**2:.1f} МБ')

## Реализация ALS

In [ ]:
class ALSRecommender:
    """
    ALS с поправками на смещения для рекомендательной системы.

    Практическая модель предсказывает рейтинг как:
        r_hat(i, j) = mu + b_u[i] + b_m[j] + U[i] @ V[j]

    Это удерживает предсказания в реалистичном диапазоне и обычно помогает ранжированию Top-K.
    """

    def __init__(
        self,
        n_factors=32,
        n_iterations=12,
        lambda_reg=0.5,
        bias_reg=10.0,
        random_state=42,
    ):
        self.n_factors = n_factors
        self.n_iterations = n_iterations
        self.lambda_reg = lambda_reg
        self.bias_reg = bias_reg
        self.random_state = random_state

        self.U = None
        self.V = None
        self.user_bias = None
        self.item_bias = None
        self.global_mean = None
        self.rating_min = None
        self.rating_max = None
        self.rmse_history = []

    def _predict_raw(self, user_idx, movie_idx):
        user_idx = np.asarray(user_idx, dtype=np.int64)
        movie_idx = np.asarray(movie_idx, dtype=np.int64)
        interaction = np.sum(self.U[user_idx] * self.V[movie_idx], axis=1)
        return (
            self.global_mean
            + self.user_bias[user_idx]
            + self.item_bias[movie_idx]
            + interaction
        )

    def predict_batch(self, user_idx, movie_idx, clip=True):
        preds = self._predict_raw(user_idx, movie_idx).astype(np.float32)
        if clip:
            preds = np.clip(preds, self.rating_min, self.rating_max)
        return preds

    def predict_all_for_user(self, user_idx, clip=True):
        scores = (
            self.global_mean
            + self.user_bias[user_idx]
            + self.item_bias
            + self.V @ self.U[user_idx]
        ).astype(np.float32)
        if clip:
            scores = np.clip(scores, self.rating_min, self.rating_max)
        return scores

    def _compute_rmse(self, R_csr):
        rows, cols = R_csr.nonzero()
        true_ratings = np.asarray(R_csr[rows, cols]).ravel().astype(np.float32)
        preds = self.predict_batch(rows, cols, clip=True)
        return float(np.sqrt(np.mean((true_ratings - preds) ** 2)))

    def fit(self, R_csr, verbose=True):
        np.random.seed(self.random_state)

        m, n = R_csr.shape
        k = self.n_factors
        lam = self.lambda_reg

        self.rating_min = float(R_csr.data.min())
        self.rating_max = float(R_csr.data.max())
        self.global_mean = float(R_csr.data.mean())

        self.U = np.random.normal(0, 0.1, size=(m, k)).astype(np.float32)
        self.V = np.random.normal(0, 0.1, size=(n, k)).astype(np.float32)
        self.user_bias = np.zeros(m, dtype=np.float32)
        self.item_bias = np.zeros(n, dtype=np.float32)

        R_csc = R_csr.tocsc()
        lambda_eye = lam * np.eye(k, dtype=np.float32)
        self.rmse_history = []

        if verbose:
            print(f'Начало обучения ALS: {m} пользователей, {n} фильмов, {k} факторов')
            print(
                f'Параметры: n_iterations={self.n_iterations}, '
                f'lambda_reg={lam}, bias_reg={self.bias_reg}'
            )
            print(f'Глобальное среднее рейтингов: {self.global_mean:.3f}')
            print('-' * 60)

        total_start = time.time()

        for iteration in range(1, self.n_iterations + 1):
            iter_start = time.time()

            for i in range(m):
                row = R_csr.getrow(i)
                rated_movie_ids = row.indices
                r_i = row.data.astype(np.float32)
                if len(rated_movie_ids) == 0:
                    continue

                V_i = self.V[rated_movie_ids]
                baseline = self.global_mean + self.user_bias[i] + self.item_bias[rated_movie_ids]
                residual = r_i - baseline
                A = V_i.T @ V_i + lambda_eye
                b = V_i.T @ residual
                self.U[i] = np.linalg.solve(A, b)

            for j in range(n):
                col = R_csc.getcol(j)
                rated_user_ids = col.indices
                r_j = col.data.astype(np.float32)
                if len(rated_user_ids) == 0:
                    continue

                U_j = self.U[rated_user_ids]
                baseline = self.global_mean + self.user_bias[rated_user_ids] + self.item_bias[j]
                residual = r_j - baseline
                A = U_j.T @ U_j + lambda_eye
                b = U_j.T @ residual
                self.V[j] = np.linalg.solve(A, b)

            for i in range(m):
                row = R_csr.getrow(i)
                rated_movie_ids = row.indices
                r_i = row.data.astype(np.float32)
                if len(rated_movie_ids) == 0:
                    continue

                interaction = np.sum(self.U[i] * self.V[rated_movie_ids], axis=1)
                residual = r_i - (
                    self.global_mean
                    + self.item_bias[rated_movie_ids]
                    + interaction
                )
                self.user_bias[i] = residual.sum() / (self.bias_reg + len(rated_movie_ids))

            for j in range(n):
                col = R_csc.getcol(j)
                rated_user_ids = col.indices
                r_j = col.data.astype(np.float32)
                if len(rated_user_ids) == 0:
                    continue

                interaction = np.sum(self.U[rated_user_ids] * self.V[j], axis=1)
                residual = r_j - (
                    self.global_mean
                    + self.user_bias[rated_user_ids]
                    + interaction
                )
                self.item_bias[j] = residual.sum() / (self.bias_reg + len(rated_user_ids))

            rmse = self._compute_rmse(R_csr)
            self.rmse_history.append(rmse)
            iter_time = time.time() - iter_start

            if verbose:
                print(
                    f'Итерация {iteration:3d}/{self.n_iterations} | '
                    f'RMSE: {rmse:.4f} | Время: {iter_time:.1f}с'
                )

        total_time = time.time() - total_start
        if verbose:
            print('-' * 60)
            print(
                f'Обучение завершено за {total_time:.1f}с | '
                f'Финальный RMSE: {self.rmse_history[-1]:.4f}'
            )
        return self

    def predict(self, user_idx, movie_idx, clip=True):
        return float(self.predict_batch([user_idx], [movie_idx], clip=clip)[0])

    def recommend(self, user_id, top_k=10, user2idx=None, idx2movie=None,
                  R_csr=None, movie_id_to_title=None):
        if user_id not in user2idx:
            return []

        user_idx = user2idx[user_id]
        pred_ratings = self.predict_all_for_user(user_idx, clip=True)

        if R_csr is not None:
            pred_ratings = pred_ratings.copy()
            watched_movie_ids = R_csr.getrow(user_idx).indices
            pred_ratings[watched_movie_ids] = -np.inf

        top_k = min(top_k, len(pred_ratings))
        candidate_idx = np.argpartition(pred_ratings, -top_k)[-top_k:]
        top_movie_indices = candidate_idx[np.argsort(pred_ratings[candidate_idx])[::-1]]

        recommendations = []
        for movie_idx in top_movie_indices:
            movie_id = idx2movie[movie_idx]
            title = movie_id_to_title.get(movie_id, f'Unknown (id={movie_id})')
            recommendations.append({
                'movie_id': movie_id,
                'title': title,
                'predicted_rating': round(float(pred_ratings[movie_idx]), 3),
            })
        return recommendations

## Обучение и визуализация

In [ ]:
# Разбиение на обучающую и тестовую части по времени
# Полное разбиение — в Секции 5. Здесь обучаем на всех данных для демонстрации.

# Сортируем по времени
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)

# 80% для обучения, 20% для теста (хронологически)
split_idx = int(len(ratings_sorted) * 0.8)
train_data = ratings_sorted.iloc[:split_idx]
test_data = ratings_sorted.iloc[split_idx:]

print(f'Обучающая часть: {len(train_data):,} записей (до {pd.to_datetime(train_data["timestamp"].max(), unit="s")})')
print(f'Тестовая часть:  {len(test_data):,} записей (от {pd.to_datetime(test_data["timestamp"].min(), unit="s")})')

# Создаём train-матрицу
train_row = train_data['user_idx'].values
train_col = train_data['movie_idx'].values
train_val = train_data['rating'].values.astype(np.float32)

R_train = sp.csr_matrix(
    (train_val, (train_row, train_col)),
    shape=(n_users, n_movies),
    dtype=np.float32
)

print(f'\nМатрица обучения: {R_train.shape}, ненулевых элементов: {R_train.nnz:,}')

In [ ]:
# Инициализация и обучение модели
model = ALSRecommender(
    n_factors=32,
    n_iterations=12,
    lambda_reg=0.5,
    bias_reg=10.0,
    random_state=42
)

model.fit(R_train, verbose=True)

In [ ]:
# График сходимости RMSE
fig, ax = plt.subplots(figsize=(9, 5))

iterations = list(range(1, len(model.rmse_history) + 1))
ax.plot(iterations, model.rmse_history, 'o-', color='steelblue',
        linewidth=2, markersize=7, label='RMSE на обучении')

# Аннотация минимума
min_idx = np.argmin(model.rmse_history)
ax.annotate(
    f'Мин: {model.rmse_history[min_idx]:.4f}',
    xy=(iterations[min_idx], model.rmse_history[min_idx]),
    xytext=(iterations[min_idx] + 0.5, model.rmse_history[min_idx] + 0.02),
    arrowprops=dict(arrowstyle='->', color='red'),
    color='red', fontsize=11
)

ax.set_xlabel('Итерация', fontsize=13)
ax.set_ylabel('RMSE на обучении', fontsize=13)
ax.set_title('Сходимость ALS: RMSE по итерациям', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(iterations)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'als_rmse_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print('График сохранён: als_rmse_curve.png')

In [ ]:
# Примеры рекомендаций для нескольких пользователей
TOP_K = 10
SAMPLE_USERS = unique_users[:5]  # берём первых 5 пользователей

for user_id in SAMPLE_USERS:
    recs = model.recommend(
        user_id=user_id,
        top_k=TOP_K,
        user2idx=user2idx,
        idx2movie=idx2movie,
        R_csr=R_train,
        movie_id_to_title=movie_id_to_title
    )
    
    user_idx = user2idx[user_id]
    n_rated = R_train.getrow(user_idx).nnz
    
    print(f'\n=== Пользователь #{user_id} (оценил {n_rated} фильмов) ===')
    print(f'  Топ-{TOP_K} рекомендаций:')
    for i, rec in enumerate(recs, 1):
        print(f'  {i:2d}. {rec["title"][:60]:<60} | Предсказание: {rec["predicted_rating"]:.3f}')

In [ ]:
# Визуализация латентного пространства (PCA-проекция факторов фильмов)
from numpy.linalg import svd

# SVD для проекции V (n_movies × k) в двумерное пространство
# Используем первые 2 правых сингулярных вектора
V_norm = model.V - model.V.mean(axis=0)  # центрирование
_, _, Vt = svd(V_norm, full_matrices=False)
V_2d = V_norm @ Vt[:2].T  # проекция на первые 2 главные компоненты

# Берём топ-200 наиболее популярных фильмов из обучающей выборки
top_movies_by_ratings = (
    train_data.groupby('movie_idx')
    .size()
    .nlargest(200)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(12, 9))

ax.scatter(
    V_2d[top_movies_by_ratings, 0],
    V_2d[top_movies_by_ratings, 1],
    alpha=0.6, s=30, color='steelblue'
)

# Подписи для топ-20 фильмов
for idx in top_movies_by_ratings[:20]:
    movie_id = idx2movie.get(idx)
    title = movie_id_to_title.get(movie_id, '')[:25] if movie_id else ''
    ax.annotate(
        title,
        (V_2d[idx, 0], V_2d[idx, 1]),
        fontsize=7, alpha=0.8
    )

ax.set_title('Латентное пространство фильмов (PCA, 2D проекция)', fontsize=14)
ax.set_xlabel('Главная компонента 1', fontsize=12)
ax.set_ylabel('Главная компонента 2', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'als_latent_space.png', dpi=100, bbox_inches='tight')
plt.show()
print('График сохранён: als_latent_space.png')

## Оценка качества

In [ ]:
# === RMSE на тестовой выборке ===
# Оцениваем только тех пользователей и фильмы, которые были видны на обучении

test_filtered = test_data[
    test_data['user_idx'].notna() &
    test_data['movie_idx'].notna()
].copy()

test_filtered = test_filtered[
    (test_filtered['user_idx'] < n_users) &
    (test_filtered['movie_idx'] < n_movies)
]

print(f'Тестовых записей для оценки: {len(test_filtered):,}')

test_user_idx = test_filtered['user_idx'].values.astype(int)
test_movie_idx = test_filtered['movie_idx'].values.astype(int)
true_ratings = test_filtered['rating'].values.astype(np.float32)

batch_size = 50_000
pred_ratings = np.zeros(len(test_user_idx), dtype=np.float32)

for start in range(0, len(test_user_idx), batch_size):
    end = min(start + batch_size, len(test_user_idx))
    pred_ratings[start:end] = model.predict_batch(
        test_user_idx[start:end],
        test_movie_idx[start:end],
        clip=True,
    )

test_rmse = np.sqrt(np.mean((true_ratings - pred_ratings) ** 2))
test_mae = np.mean(np.abs(true_ratings - pred_ratings))

print(f'\n=== Метрики на тестовой выборке ===')
print(f'RMSE: {test_rmse:.4f}')
print(f'MAE:  {test_mae:.4f}')
print(f'\nRMSE на обучении: {model.rmse_history[-1]:.4f}')
print(f'Разность (train - test): {model.rmse_history[-1] - test_rmse:.4f}')

In [ ]:
# === Precision@K и Recall@K ===
# Определение: рейтинг >= порога считается "релевантным"

RELEVANCE_THRESHOLD = 4.0
K_VALUES = [5, 10, 20]

def compute_precision_recall_at_k(model, test_data, R_train, user2idx, idx2movie,
                                  n_users, n_movies, k_values, threshold=4.0,
                                  n_sample_users=200):
    """
    Вычисляет Precision@K и Recall@K для выборки пользователей.
    """
    results = {k: {'precision': [], 'recall': []} for k in k_values}

    test_grouped = test_data.groupby('user_idx')
    test_user_indices = list(test_grouped.groups.keys())

    np.random.seed(42)
    sample_indices = np.random.choice(
        test_user_indices,
        size=min(n_sample_users, len(test_user_indices)),
        replace=False
    )

    for user_idx in sample_indices:
        user_idx = int(user_idx)
        user_test = test_grouped.get_group(user_idx)

        relevant_movies = set(
            user_test[user_test['rating'] >= threshold]['movie_idx'].astype(int).tolist()
        )
        if len(relevant_movies) == 0:
            continue

        pred_scores = model.predict_all_for_user(user_idx, clip=True)
        watched = R_train.getrow(user_idx).indices
        pred_scores = pred_scores.copy()
        pred_scores[watched] = -np.inf

        for k in k_values:
            top_k = set(np.argsort(pred_scores)[::-1][:k].tolist())
            hits = len(top_k & relevant_movies)
            results[k]['precision'].append(hits / k)
            results[k]['recall'].append(hits / len(relevant_movies))

    summary = {}
    for k in k_values:
        prec_list = results[k]['precision']
        rec_list = results[k]['recall']
        summary[k] = {
            'precision_at_k': np.mean(prec_list) if prec_list else 0.0,
            'recall_at_k': np.mean(rec_list) if rec_list else 0.0,
            'n_users_evaluated': len(prec_list)
        }
    return summary


print('Вычисление Precision@K и Recall@K...')
t0 = time.time()

pr_results = compute_precision_recall_at_k(
    model=model,
    test_data=test_filtered,
    R_train=R_train,
    user2idx=user2idx,
    idx2movie=idx2movie,
    n_users=n_users,
    n_movies=n_movies,
    k_values=K_VALUES,
    threshold=RELEVANCE_THRESHOLD,
    n_sample_users=200
)

print(f'Готово за {time.time()-t0:.1f} с')
print(f'\nПорог релевантности: рейтинг >= {RELEVANCE_THRESHOLD}')
print()
print(f'{"K":>5} | {"Precision@K":>12} | {"Recall@K":>10} | {"Пользователей":>14}')
print('-' * 52)
for k in K_VALUES:
    r = pr_results[k]
    print(f'{k:>5} | {r["precision_at_k"]:>12.4f} | {r["recall_at_k"]:>10.4f} | {r["n_users_evaluated"]:>14}')

In [ ]:
# Итоговая сводка и визуализация метрик
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# График 1: RMSE сходимость (итоговый)
ax = axes[0]
ax.plot(range(1, len(model.rmse_history) + 1), model.rmse_history,
        'o-', color='steelblue', linewidth=2, markersize=6)
ax.axhline(y=test_rmse, color='red', linestyle='--', label=f'RMSE на тесте: {test_rmse:.4f}')
ax.set_title('RMSE: обучение и тест', fontsize=12)
ax.set_xlabel('Итерация')
ax.set_ylabel('RMSE')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# График 2: Precision@K
ax = axes[1]
k_list = K_VALUES
prec_list = [pr_results[k]['precision_at_k'] for k in k_list]
bars = ax.bar([str(k) for k in k_list], prec_list, color='steelblue', edgecolor='black', alpha=0.8)
for bar, val in zip(bars, prec_list):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_title('Precision@K', fontsize=12)
ax.set_xlabel('K')
ax.set_ylabel('Precision')
ax.grid(True, alpha=0.3, axis='y')

# График 3: Recall@K
ax = axes[2]
rec_list = [pr_results[k]['recall_at_k'] for k in k_list]
bars = ax.bar([str(k) for k in k_list], rec_list, color='coral', edgecolor='black', alpha=0.8)
for bar, val in zip(bars, rec_list):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_title('Recall@K', fontsize=12)
ax.set_xlabel('K')
ax.set_ylabel('Recall')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Итоговые метрики ALS рекомендательной системы', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'als_final_metrics.png', dpi=100, bbox_inches='tight')
plt.show()
print('График сохранён: als_final_metrics.png')

# Итоговая сводка
print('\n' + '='*55)
print('ИТОГОВАЯ СВОДКА')
print('='*55)
print(f'Датасет:       MovieLens ml-32m (первые 500k записей)')
print(f'Пользователей: {n_users:,}')
print(f'Фильмов:       {n_movies:,}')
print(f'Число факторов: {model.n_factors}')
print(f'Итераций ALS:  {model.n_iterations}')
print(f'lambda_reg:    {model.lambda_reg}')
print(f'RMSE на обучении: {model.rmse_history[-1]:.4f}')
print(f'RMSE на тесте:   {test_rmse:.4f}')
print(f'MAE на тесте:    {test_mae:.4f}')
print('-'*55)
for k in K_VALUES:
    r = pr_results[k]
    print(f'Precision@{k:<3}: {r["precision_at_k"]:.4f}   Recall@{k:<3}: {r["recall_at_k"]:.4f}')
print('='*55)